# Stage 4 — Feature Engineering

## Purpose
Transform the cleaned dataset into the exact numerical format required
by machine learning models. Every transformation is fit ONLY on training
data to prevent data leakage into the test set.

## Operations Performed
| Step | Operation | Reason |
|------|-----------|--------|
| 1 | One-hot encode multi-class columns | Models need numbers, not text |
| 2 | Drop first dummy column per group | Prevent dummy variable trap |
| 3 | Separate X (features) and y (target) | Standard ML convention |
| 4 | Train/test split (80/20 stratified) | Honest model evaluation |
| 5 | Scale numeric features | Equal contribution across features |
| 6 | Save all outputs | Stage 5 loads these directly |

## Critical Rule — No Data Leakage
The scaler is fit ONLY on X_train, then applied to both X_train and X_test.
Fitting on X_test would give the model illegal "future knowledge."

In [1]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import StandardScaler

print("All libraries imported Successfully.")

All libraries imported Successfully.


In [2]:
# Load cleaned dataset
DATA_PATH      = '../data/processed/churn_cleaned.csv'
PROCESSED_PATH = '../data/processed/'
MODELS_PATH    = '../models/'

os.makedirs(PROCESSED_PATH, exist_ok=True)
os.makedirs(MODELS_PATH,    exist_ok=True)

df = pd.read_csv(DATA_PATH)

print(f"Loaded: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"\nColumn types:")
print(df.dtypes.value_counts())

Loaded: 7032 rows × 20 columns
Missing values: 0

Column types:
str        10
int64       8
float64     2
Name: count, dtype: int64


In [3]:
# Identify which columns need enoding

test_cols    = df.select_dtypes(include='object').columns.tolist()
numeric_cols = df.select_dtypes(include='number').columns.tolist()

print("Text columns requiring one-hot encoding:")
for col in test_cols:
    vals = df[col].unique().tolist()
    print(f" {col:<25} - {len(vals)} categories: {vals}")

print(f"\nNumeric columns (already encoded or continuous):")
print(f" {numeric_cols}") 

Text columns requiring one-hot encoding:
 MultipleLines             - 3 categories: ['No phone service', 'No', 'Yes']
 InternetService           - 3 categories: ['DSL', 'Fiber optic', 'No']
 OnlineSecurity            - 3 categories: ['No', 'Yes', 'No internet service']
 OnlineBackup              - 3 categories: ['Yes', 'No', 'No internet service']
 DeviceProtection          - 3 categories: ['No', 'Yes', 'No internet service']
 TechSupport               - 3 categories: ['No', 'Yes', 'No internet service']
 StreamingTV               - 3 categories: ['No', 'Yes', 'No internet service']
 StreamingMovies           - 3 categories: ['No', 'Yes', 'No internet service']
 Contract                  - 3 categories: ['Month-to-month', 'One year', 'Two year']
 PaymentMethod             - 4 categories: ['Electronic check', 'Mailed check', 'Bank transfer (automatic)', 'Credit card (automatic)']

Numeric columns (already encoded or continuous):
 ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'ten

C:\Users\Vishn\AppData\Local\Temp\ipykernel_22688\2480655745.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  test_cols    = df.select_dtypes(include='object').columns.tolist()


In [4]:
# Apply one-hot encoding to all text columns

rows_before  = df.shape[0]
cols_before  = df.shape[1]

df_encoded = pd.get_dummies(df, columns=test_cols,
                            drop_first=True, dtype=int)

rows_after = df_encoded.shape[0]
cols_after = df_encoded.shape[1]

print(f"Shape before encoding: {rows_before} rows × {cols_before} columns")
print(f"Shape after encoding : {rows_after} rows × {cols_after} columns")
print(f"New columns added    : {cols_after - cols_before}")
print()
for i, col in enumerate(df_encoded.columns, 1):
    print(f"  {i:>2}. {col}")

Shape before encoding: 7032 rows × 20 columns
Shape after encoding : 7032 rows × 31 columns
New columns added    : 11

   1. gender
   2. SeniorCitizen
   3. Partner
   4. Dependents
   5. tenure
   6. PhoneService
   7. PaperlessBilling
   8. MonthlyCharges
   9. TotalCharges
  10. Churn
  11. MultipleLines_No phone service
  12. MultipleLines_Yes
  13. InternetService_Fiber optic
  14. InternetService_No
  15. OnlineSecurity_No internet service
  16. OnlineSecurity_Yes
  17. OnlineBackup_No internet service
  18. OnlineBackup_Yes
  19. DeviceProtection_No internet service
  20. DeviceProtection_Yes
  21. TechSupport_No internet service
  22. TechSupport_Yes
  23. StreamingTV_No internet service
  24. StreamingTV_Yes
  25. StreamingMovies_No internet service
  26. StreamingMovies_Yes
  27. Contract_One year
  28. Contract_Two year
  29. PaymentMethod_Credit card (automatic)
  30. PaymentMethod_Electronic check
  31. PaymentMethod_Mailed check


In [5]:
# Verify the encoding is compltely clean

remaining_text = df_encoded.select_dtypes(include='object').columns.tolist()
total_nulls    = df_encoded.isnull().sum().sum()
all_numeric    = all(df_encoded.dtypes != 'object')

print("Encoding Verification:")
print(f"  Remaining text columns : {remaining_text if remaining_text else 'None'}")
print(f"  All columns numeric    : {all_numeric}")
print()

if not remaining_text and total_nulls == 0 and all_numeric:
    print("all checks passed. Dataset is fully numeric and clean.")
else:
    print("WARNING: One or more checks failed. Investigate before continuing.")

Encoding Verification:
  Remaining text columns : None
  All columns numeric    : True

all checks passed. Dataset is fully numeric and clean.


In [6]:
# Split into X (features) and y (target)

X = df_encoded.drop(columns=['Churn'])
y = df_encoded['Churn']

print(f"Feature matrix X: {X.shape[0]} rows × {X.shape[1]} features")
print(f"Target vector  y: {y.shape[0]} values")
print()
print(f"Churn distribution in y:")
print(f"  Retained (0): {(y == 0).sum():,}  ({(y == 0).mean()*100:.1f}%)")
print(f"  Churned  (1): {(y == 1).sum():,}  ({(y == 1).mean()*100:.1f}%)")
print()
print("Feature columns in X:")
for i, col in enumerate(X.columns, 1):
    print(f"  {i:>2}. {col}")


Feature matrix X: 7032 rows × 30 features
Target vector  y: 7032 values

Churn distribution in y:
  Retained (0): 5,163  (73.4%)
  Churned  (1): 1,869  (26.6%)

Feature columns in X:
   1. gender
   2. SeniorCitizen
   3. Partner
   4. Dependents
   5. tenure
   6. PhoneService
   7. PaperlessBilling
   8. MonthlyCharges
   9. TotalCharges
  10. MultipleLines_No phone service
  11. MultipleLines_Yes
  12. InternetService_Fiber optic
  13. InternetService_No
  14. OnlineSecurity_No internet service
  15. OnlineSecurity_Yes
  16. OnlineBackup_No internet service
  17. OnlineBackup_Yes
  18. DeviceProtection_No internet service
  19. DeviceProtection_Yes
  20. TechSupport_No internet service
  21. TechSupport_Yes
  22. StreamingTV_No internet service
  23. StreamingTV_Yes
  24. StreamingMovies_No internet service
  25. StreamingMovies_Yes
  26. Contract_One year
  27. Contract_Two year
  28. PaymentMethod_Credit card (automatic)
  29. PaymentMethod_Electronic check
  30. PaymentMethod_Mai

In [7]:
# Perform the train/test split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.2,
    random_state = 42,
    stratify     = y 
)

print("Train / test split results:")
print(f" X_train  : {X_train.shape[0]:,} rows × {X_train.shape[1]} features")
print(f" X_test   : {X_test.shape[0]:,} rows × {X_test.shape[1]} features")
print()
print("Churn distribution - verifying stratification worked:")
print(f"  Training set churn rate : {y_train.mean()*100:.2f}%")
print(f"  Test set     churn rate : {y_test.mean()*100:.2f}%")
print(f"  Original     churn rate : {y.mean()*100:.2f}%")
print()

# All three percentages should be within 0.1% of each other
if abs(y_train.mean() - y_test.mean()) < 0.005:
    print("Stratification confirmed: churn rates are consistent across splits.")
else:
    print("Warning: Churn rates differ significantly between splits.")

Train / test split results:
 X_train  : 5,625 rows × 30 features
 X_test   : 1,407 rows × 30 features

Churn distribution - verifying stratification worked:
  Training set churn rate : 26.58%
  Test set     churn rate : 26.58%
  Original     churn rate : 26.58%

Stratification confirmed: churn rates are consistent across splits.


In [8]:
# Identify numeric columns to scale 

cols_to_scale = ['tenure', 'MonthlyCharges', 'TotalCharges']

print("Columns to be scaled:")
for col in cols_to_scale:
    print(f"  {col}")
    print(f"    Train - min: {X_train[col].min():.2f} "
          f"max: {X_train[col].max():.2f} "
          f"mean: {X_train[col].mean():.2f}")
print()
print("All other columns are binary (0/1) and do not need scaling.")

Columns to be scaled:
  tenure
    Train - min: 1.00 max: 72.00 mean: 32.56
  MonthlyCharges
    Train - min: 18.40 max: 118.65 mean: 65.00
  TotalCharges
    Train - min: 18.80 max: 8684.80 mean: 2301.84

All other columns are binary (0/1) and do not need scaling.


In [9]:
# Fit and apply StandardScaler

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# Fit only on training data, then transform both sets
X_train_scaled[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test_scaled[cols_to_scale]  = scaler.transform(X_test[cols_to_scale])

print("Scaling applied. Verification - scaled column statistics:")
print()
print("Training set (should be mean ≈ 0, std ≈ 1):")
print(X_train_scaled[cols_to_scale].describe().round(3).loc[['mean', 'std', 'min', 'max']])
print()
print("Test set (mean will not be exactly 0 - that is correct and expected):")
print(X_test_scaled[cols_to_scale].describe().round(3).loc[['mean', 'std', 'min', 'max']])

Scaling applied. Verification - scaled column statistics:

Training set (should be mean ≈ 0, std ≈ 1):
      tenure  MonthlyCharges  TotalCharges
mean  -0.000           0.000         0.000
std    1.000           1.000         1.000
min   -1.286          -1.548        -1.003
max    1.607           1.782         2.805

Test set (mean will not be exactly 0 - that is correct and expected):
      tenure  MonthlyCharges  TotalCharges
mean  -0.029          -0.033        -0.041
std    1.001           0.996         0.980
min   -1.286          -1.553        -1.003
max    1.607           1.785         2.800


In [10]:
# Reset index on all four datasets

X_train_scaled = X_train_scaled.reset_index(drop=True)
X_test_scaled  = X_test_scaled.reset_index(drop=True)
y_train        = y_train.reset_index(drop=True)
y_test         = y_test.reset_index(drop=True)

print("Index reset complete.")
print(f"X_train index range: {X_train_scaled.index[0]} to {X_train_scaled.index[-1]}")
print(f"X_test index range: {X_test_scaled.index[0]} to {X_test_scaled.index[-1]}")

Index reset complete.
X_train index range: 0 to 5624
X_test index range: 0 to 1406


In [11]:
# Save four feature matrices as CSV files

output_files = {
    'x_train.csv' : X_train_scaled,
    'X_test.csv'  : X_test_scaled,
    'y_train.csv' : y_train.to_frame(),
    'y_test.csv'  : y_test.to_frame(),
}

for filename, data in output_files.items():
    filepath = os.path.join(PROCESSED_PATH, filename)
    data.to_csv(filepath, index=False)
    size_kb = os.path.getsize(filepath) / 1024
    print(f"Saved: {filename:<20} {data.shape} ({size_kb:.1f} KB)")

    print()
    print("All four files saved to data/processed/")

Saved: x_train.csv          (5625, 30) (625.4 KB)

All four files saved to data/processed/
Saved: X_test.csv           (1407, 30) (156.9 KB)

All four files saved to data/processed/
Saved: y_train.csv          (5625, 1) (16.5 KB)

All four files saved to data/processed/
Saved: y_test.csv           (1407, 1) (4.1 KB)

All four files saved to data/processed/


In [12]:
# Save the scaler object using joblib

scaler_path = os.path.join(MODELS_PATH, 'scaler.pkl')
joblib.dump(scaler, scaler_path)

print(f"Scaler saved to: {scaler_path}")
print()
print(f"Scaler parameters (learned from training data):")
for col, mean, std in zip(cols_to_scale,
                           scaler.mean_,
                           scaler.scale_):
    print(f" {col:<20} mean={mean:.4f}  std={std:.4f}")

Scaler saved to: ../models/scaler.pkl

Scaler parameters (learned from training data):
 tenure               mean=32.5623  std=24.5402
 MonthlyCharges       mean=64.9993  std=30.1060
 TotalCharges         mean=2301.8395  std=2275.3838


In [13]:
text_cols = df.select_dtypes(include='object').columns.tolist()
print(f"text_cols restored: {text_cols}")

text_cols restored: ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']


C:\Users\Vishn\AppData\Local\Temp\ipykernel_22688\238124087.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = df.select_dtypes(include='object').columns.tolist()


In [14]:
# Complete Stage 4 validation report

print("=" * 60)
print("  STAGE 4 — FEATURE ENGINEERING COMPLETE")
print("=" * 60)

print(f"""
  INPUT
    File  : data/processed/churn_cleaned.csv
    Shape : 7,032 rows × 20 columns

  ENCODING
    Text columns one-hot encoded : {len(text_cols)}
    Dummy trap columns dropped   : {len(text_cols)} (one per group)
    Final feature count          : {X_train_scaled.shape[1]}

  SPLIT
    Training rows : {len(X_train_scaled):,}  ({len(X_train_scaled)/len(X)*100:.0f}%)
    Test rows     : {len(X_test_scaled):,}   ({len(X_test_scaled)/len(X)*100:.0f}%)
    Train churn % : {y_train.mean()*100:.2f}%
    Test  churn % : {y_test.mean()*100:.2f}%

  SCALING
    Method   : StandardScaler (mean=0, std=1)
    Fit on   : X_train ONLY (no data leakage)
    Columns  : tenure, MonthlyCharges, TotalCharges

  OUTPUTS SAVED
    data/processed/X_train.csv   — {X_train_scaled.shape}
    data/processed/X_test.csv    — {X_test_scaled.shape}
    data/processed/y_train.csv   — {y_train.shape}
    data/processed/y_test.csv    — {y_test.shape}
    models/scaler.pkl            — StandardScaler object
""")
print("=" * 60)
print("  Ready for Stage 5 — Model Training")
print("=" * 60)

  STAGE 4 — FEATURE ENGINEERING COMPLETE

  INPUT
    File  : data/processed/churn_cleaned.csv
    Shape : 7,032 rows × 20 columns

  ENCODING
    Text columns one-hot encoded : 10
    Dummy trap columns dropped   : 10 (one per group)
    Final feature count          : 30

  SPLIT
    Training rows : 5,625  (80%)
    Test rows     : 1,407   (20%)
    Train churn % : 26.58%
    Test  churn % : 26.58%

  SCALING
    Method   : StandardScaler (mean=0, std=1)
    Fit on   : X_train ONLY (no data leakage)
    Columns  : tenure, MonthlyCharges, TotalCharges

  OUTPUTS SAVED
    data/processed/X_train.csv   — (5625, 30)
    data/processed/X_test.csv    — (1407, 30)
    data/processed/y_train.csv   — (5625,)
    data/processed/y_test.csv    — (1407,)
    models/scaler.pkl            — StandardScaler object

  Ready for Stage 5 — Model Training


## Stage 4 Complete

All feature matrices are saved and ready for model training.

### Key Decisions Made
- **One-hot encoding** applied to all 7 multi-class text columns
- **drop_first=True** prevents the dummy variable trap
- **80/20 stratified split** preserves the 26.5% churn rate in both sets
- **StandardScaler** fit on training data only — no data leakage
- **TotalCharges retained** — tree-based models handle correlated features natively

### Next Step
Stage 5 loads X_train, X_test, y_train, y_test directly from CSV files
and trains Logistic Regression, Random Forest, and XGBoost models.